Model Comparison & Feature Engineering

In [0]:
import pandas as pd

df = spark.table("workspace.ecommerce.gold_product_performance").toPandas()
df.head()


,product_id,views,cart_adds,purchases,revenue
0,1005159,57475,1579,1063,238752.03
1,5701087,4819,0,80,4118.40
2,38300497,14,0,0,NaN
3,41100005,458,0,1,68.21
4,14700174,131,0,0,NaN


In [0]:
X = df[["views", "cart_adds"]]
y = df["purchases"]


In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    "linear_regression": LinearRegression(),
    "decision_tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "random_forest": RandomForestRegressor(n_estimators=50, random_state=42)
}

for name, model in models.items():
    with mlflow.start_run(run_name=f"{name}_model"):

        mlflow.log_param("model_type", name)

        model.fit(X_train, y_train)
        r2 = model.score(X_test, y_test)

        mlflow.log_metric("r2_score", r2)
        mlflow.sklearn.log_model(model, "model")

        print(f"{name}: R² = {r2:.4f}")


2026/01/21 14:55:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


linear_regression: R² = 0.9817


2026/01/21 14:55:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


decision_tree: R² = 0.9349


2026/01/21 14:55:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


random_forest: R² = 0.9633


In [0]:
rf = models["random_forest"]

importance = rf.feature_importances_
list(zip(X.columns, importance))


[('views', np.float64(0.43535496807406276)),
 ('cart_adds', np.float64(0.5646450319259372))]

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression as SparkLR

spark_df = spark.table("workspace.ecommerce.gold_product_performance")

assembler = VectorAssembler(
    inputCols=["views", "cart_adds"],
    outputCol="features"
)

lr = SparkLR(
    featuresCol="features",
    labelCol="purchases"
)

pipeline = Pipeline(stages=[assembler, lr])


In [0]:
train, test = spark_df.randomSplit([0.8, 0.2], seed=42)
pipeline_model = pipeline.fit(train)
